# 🔬 DermaFusion-AI — Kaggle Training Notebook
## EVA-02 Large + ConvNeXt V2 | Dual-Branch Fusion | SOTA 2026

### ⚙️ Before Running:
1. **Add Data** (right panel) — attach these 4 datasets (already done ✅)
2. Set **Accelerator → GPU T4 x2** in the right panel
3. Set **Persistence → Files** so weights survive between sessions
4. Run cells **top to bottom**

## Step 0: Check What Kaggle Input Folders Exist

In [ ]:
import os
print('📁 Available /kaggle/input/ folders:')
for folder in sorted(os.listdir('/kaggle/input/')):
    print(f'   /kaggle/input/{folder}')

## Step 1: Clone Your GitHub Repo

In [ ]:
import os

if not os.path.exists('/kaggle/working/DermaFusion-AI'):
    !git clone https://github.com/ai-with-abdullah/DermaFusion-AI.git /kaggle/working/DermaFusion-AI
else:
    print('✅ Repo already cloned — pulling latest changes')
    !cd /kaggle/working/DermaFusion-AI && git pull

os.chdir('/kaggle/working/DermaFusion-AI')
print('Working directory:', os.getcwd())

## Step 2: Install Dependencies

In [ ]:
!pip install -q timm>=0.9.12 albumentations>=1.3.1 opencv-python-headless scikit-learn scipy tqdm
print('✅ Dependencies installed')

## Step 3: Verify GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'✅ GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')
else:
    print('❌ No GPU found — go to Settings > Accelerator > GPU T4 x2')

## Step 4: Auto-Link Kaggle Datasets → data/ folder
This cell **automatically finds** your dataset folders regardless of the exact slug name Kaggle assigned.

In [ ]:
import os

os.makedirs('/kaggle/working/DermaFusion-AI/data', exist_ok=True)
kaggle_input = '/kaggle/input'
available = os.listdir(kaggle_input)

def find_and_link(possible_slugs, dst_name, label):
    """Try multiple possible slug names and symlink the first match found."""
    dst = f'/kaggle/working/DermaFusion-AI/data/{dst_name}'
    if os.path.exists(dst):
        print(f'✅ {label}: already linked at data/{dst_name}')
        return True
    for slug in possible_slugs:
        src = f'{kaggle_input}/{slug}'
        if os.path.exists(src):
            os.symlink(src, dst)
            print(f'✅ {label}: linked from /kaggle/input/{slug}')
            return True
    print(f'❌ {label}: NOT FOUND — tried: {possible_slugs}')
    print(f'   Available folders: {available}')
    return False

# ── HAM10000 ────────────────────────────────────────────────────────────────
find_and_link(
    ['skin-cancer-mnist-ham10000', 'ham10000', 'skin-cancer-mnist'],
    'ham10000', 'HAM10000'
)

# ── ISIC 2019 ───────────────────────────────────────────────────────────────
find_and_link(
    ['skin-lesion-images-for-melanoma-classification', 'isic-2019',
     'isic2019', 'skin-lesion-images-for-melanoma'],
    'isic_2019', 'ISIC 2019'
)

# ── ISIC 2020 ───────────────────────────────────────────────────────────────
find_and_link(
    ['siim-isic-melanoma-classification', 'isic-2020', 'melanoma-classification',
     'siim-isic-melanoma'],
    'isic_2020', 'ISIC 2020'
)

# ── ISIC 2024 ───────────────────────────────────────────────────────────────
find_and_link(
    ['isic-2024-challenge', 'isic-2024', 'skin-cancer-detection-3d-tbp',
     'isic2024'],
    'isic_2024', 'ISIC 2024'
)

print('\n📁 data/ contents:', os.listdir('/kaggle/working/DermaFusion-AI/data/'))

## Step 5: Phase 1 — Train Segmentation Model (25 epochs)
Trains the **Swin-UNet** to generate lesion masks. Must run before classifier training.

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')

!python train_segmentation.py 2>&1 | tee /kaggle/working/train_segmentation.log
print('\n✅ Segmentation training complete!')

## Step 6: Phase 2 — Train Classifier (25 epochs)
Trains the **EVA-02 Large + ConvNeXt V2** dual-branch fusion classifier.

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')

!python train_classifier.py 2>&1 | tee /kaggle/working/train_classifier.log
print('\n✅ Classifier training complete!')

## Step 7: Save Weights to Kaggle Output

In [ ]:
import shutil, os

weights_src = '/kaggle/working/DermaFusion-AI/outputs/weights/'
output_dir  = '/kaggle/working/'

if os.path.exists(weights_src):
    for f in os.listdir(weights_src):
        if f.endswith('.pth'):
            shutil.copy(os.path.join(weights_src, f), output_dir)
            size_mb = os.path.getsize(os.path.join(output_dir, f)) / 1e6
            print(f'✅ Saved: {f}  ({size_mb:.0f} MB)')
    print('\n📦 Weights saved. Download from the Output tab on the right.')
else:
    print('❌ No weights found — did training complete?')

## Step 8: Full Evaluation (Metrics + GradCAM++ XAI)

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')

!python evaluate.py 2>&1 | tee /kaggle/working/evaluate.log
print('\n✅ Evaluation complete! Check the Output tab for results.')